In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data/raw")
WEATHER_CSV = DATA_DIR / "weatherAUS.csv"
SOLAR_DIR = (
    DATA_DIR / "Australia_GISdata_LTAy_AvgDailyTotals_GlobalSolarAtlas-v2_GEOTIFF"
)

df_all = pd.read_csv(WEATHER_CSV, parse_dates=["Date"])
df_all["Year"] = df_all["Date"].dt.year
print(
    f"Total rows: {len(df_all)} | Date range: {df_all['Date'].min().date()} to {df_all['Date'].max().date()}"
)

In [ ]:
missing_pct = df_all.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

missing_pct.plot(kind="bar", figsize=(10, 4), color="steelblue")
plt.title("Missing Value % Across Full Dataset")
plt.ylabel("Missing %")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
sunshine_by_year = (
    df_all.groupby("Year")["Sunshine"].apply(lambda x: x.isnull().mean() * 100).round(1)
)

cities_with_sunshine = df_all.groupby("Year").apply(
    lambda x: (
        x.groupby("Location")["Sunshine"].apply(lambda s: s.isnull().mean() < 0.5)
    ).sum()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sunshine_by_year.plot(kind="bar", ax=axes[0], color="orange")
axes[0].set_title("Sunshine Missing % by Year")
axes[0].set_ylabel("Missing %")
axes[0].set_xlabel("")

cities_with_sunshine.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Cities with Usable Sunshine by Year")
axes[1].set_ylabel("City Count")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

print(
    pd.DataFrame(
        {"missing_pct": sunshine_by_year, "cities_with_sunshine": cities_with_sunshine}
    )
)

In [ ]:
TRAIN_YEAR = 2009
HOLDOUT_YEAR = 2010

df_train = df_all[df_all["Year"] == TRAIN_YEAR].copy()
df_holdout = df_all[df_all["Year"] == HOLDOUT_YEAR].copy()

for df in [df_train, df_holdout]:
    df["Month"] = df["Date"].dt.month
    df["DayOfYear"] = df["Date"].dt.dayofyear

print(f"Train   : {df_train.shape}  | cities: {df_train['Location'].nunique()}")
print(f"Holdout : {df_holdout.shape} | cities: {df_holdout['Location'].nunique()}")

for name, df in [
    (f"{TRAIN_YEAR} Train", df_train),
    (f"{HOLDOUT_YEAR} Holdout", df_holdout),
]:
    missing = (df.isnull().mean() * 100).round(1)
    missing = missing[missing > 0].sort_values(ascending=False)
    print(f"\n=== {name} ===")
    print(missing)

df_holdout_raw = df_holdout.copy()  # preserve before encoding

In [ ]:
DROP_COLS = ["Date", "Year", "RainTomorrow"]

df_train = df_train.drop(columns=DROP_COLS)
df_holdout = df_holdout.drop(columns=DROP_COLS)

print(f"Train shape   : {df_train.shape}")
print(f"Holdout shape : {df_holdout.shape}")
print(f"Remaining cols: {df_train.columns.tolist()}")

In [ ]:
check_cols = ["MaxTemp", "Rainfall", "Humidity9am", "Cloud9am", "Sunshine"]
check_cols = [c for c in check_cols if c in df_train.columns]

stats = pd.concat(
    [
        df_train[check_cols]
        .describe()
        .T[["mean", "std", "min", "max"]]
        .add_suffix("_train"),
        df_holdout[check_cols]
        .describe()
        .T[["mean", "std", "min", "max"]]
        .add_suffix("_holdout"),
    ],
    axis=1,
).round(2)

# Reorder columns to interleave train/holdout
stats = stats[
    [
        c
        for pair in zip(
            [c for c in stats.columns if "_train" in c],
            [c for c in stats.columns if "_holdout" in c],
        )
        for c in pair
    ]
]

print(stats)

In [ ]:
from geopy.geocoders import Nominatim
import re
import time

geolocator = Nominatim(user_agent="solar_assessment")


def normalize_city(city: str) -> str:
    # Insert space before each uppercase letter that follows a lowercase letter
    # e.g. "CoffsHarbour" → "Coffs Harbour", "PearceRAAF" → "Pearce RAAF"
    return re.sub(r"([a-z])([A-Z])", r"\1 \2", city)


def geocode_city(city: str) -> tuple:
    for query in [f"{normalize_city(city)}, Australia", f"{city}, Australia"]:
        try:
            location = geolocator.geocode(query)
            time.sleep(1)
            if location:
                return (location.latitude, location.longitude)
        except Exception:
            pass
    return (None, None)


cities = df_train["Location"].unique()
city_coords = {city: geocode_city(city) for city in cities}

failed = [c for c, coords in city_coords.items() if None in coords]
print(f"Geocoded: {len(cities) - len(failed)}/{len(cities)}")
if failed:
    print(f"Failed: {failed}")

In [ ]:
import rasterio
from rasterio.transform import rowcol

PVOUT_TIF = SOLAR_DIR / "PVOUT.tif"

with rasterio.open(PVOUT_TIF) as src:
    data = src.read(1)
    data = np.where(data == src.nodata, np.nan, data)

plt.figure(figsize=(10, 6))
plt.imshow(data, cmap="YlOrRd", aspect="auto")
plt.colorbar(label="PVOUT (kWh/kWp/day)")
plt.title("GSA PVOUT Raster — Australia")
plt.tight_layout()
plt.show()

In [ ]:
def extract_pvout(lat: float, lon: float, src) -> float:
    row, col = rowcol(src.transform, lon, lat)
    if row < 0 or col < 0 or row >= src.height or col >= src.width:
        return None
    value = src.read(1)[row, col]
    return None if value != value else float(value)  # NaN check


with rasterio.open(PVOUT_TIF) as src:
    print(f"CRS   : {src.crs}")
    print(f"Bounds: {src.bounds}")

    city_pvout = {
        city: extract_pvout(lat, lon, src) for city, (lat, lon) in city_coords.items()
    }

pvout_df = pd.Series(city_pvout).sort_values(ascending=False)
print(f"\nPVOUT extracted: {pvout_df.notna().sum()}/{len(pvout_df)} cities")
print(f"Range: {pvout_df.min():.2f} - {pvout_df.max():.2f} kWh/kWp/day")
print(pvout_df)

In [ ]:
valid = {city: pvout for city, pvout in city_pvout.items() if pvout is not None}
lats = [city_coords[c][0] for c in valid]
lons = [city_coords[c][1] for c in valid]
vals = list(valid.values())

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    lons, lats, c=vals, cmap="YlOrRd", s=100, edgecolors="black", linewidths=0.5
)
plt.colorbar(scatter, label="PVOUT (kWh/kWp/day)")

for city, pvout in valid.items():
    lat, lon = city_coords[city]
    plt.annotate(city, (lon, lat), fontsize=6, ha="left", va="bottom")

plt.title("GSA Yearly PVOUT per City")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# Map PVOUT to each row by city
df_train["PVOUT"] = df_train["Location"].map(city_pvout)
df_holdout["PVOUT"] = df_holdout["Location"].map(city_pvout)

# Drop rows where PVOUT is None (NorfolkIsland)
df_train = df_train.dropna(subset=["PVOUT"])
df_holdout = df_holdout.dropna(subset=["PVOUT"])

print(f"Train shape   : {df_train.shape} | cities: {df_train['Location'].nunique()}")
print(
    f"Holdout shape : {df_holdout.shape} | cities: {df_holdout['Location'].nunique()}"
)
print(f"\nPVOUT distribution:")
print(df_train["PVOUT"].describe().round(3))

In [ ]:
from sklearn.preprocessing import LabelEncoder

FEATURE_COLS = [c for c in df_train.columns if c not in ["Location", "PVOUT"]]
CAT_COLS = ["WindGustDir", "WindDir9am", "WindDir3pm", "RainToday"]

# Label encode categoricals
encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    combined = pd.concat([df_train[col], df_holdout[col]]).astype(str)
    le.fit(combined)
    df_train[col] = le.transform(df_train[col].astype(str))
    df_holdout[col] = le.transform(df_holdout[col].astype(str))
    encoders[col] = le

X_train = df_train[FEATURE_COLS]
y_train = df_train["PVOUT"]
X_test = df_holdout[FEATURE_COLS]
y_test = df_holdout["PVOUT"]

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Features: {FEATURE_COLS}")

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    enable_categorical=False,
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=100)

y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nRMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

In [ ]:
importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(
    ascending=False
)

importance.plot(kind="bar", figsize=(12, 4), color="steelblue")
plt.title("XGBoost Feature Importance")
plt.ylabel("Importance Score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(importance.round(4))

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.1, s=5, color="steelblue")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("Actual PVOUT")
plt.ylabel("Predicted PVOUT")
plt.title(f"Predicted vs Actual (R²={r2:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": 42,
        "n_jobs": -1,
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    return r2_score(y_test, model.predict(X_test))


study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"Best R²    : {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
best_model = xgb.XGBRegressor(**study.best_params, random_state=42, n_jobs=-1)
best_model.fit(X_train, y_train)

y_pred_tuned = best_model.predict(X_test)
rmse_tuned = mean_squared_error(y_test, y_pred_tuned) ** 0.5
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print(f"Baseline → Tuned")
print(f"RMSE : {rmse:.4f} → {rmse_tuned:.4f}")
print(f"MAE  : {mae:.4f} → {mae_tuned:.4f}")
print(f"R²   : {r2:.4f} → {r2_tuned:.4f}")

In [ ]:
import joblib

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

best_model.save_model(PROCESSED_DIR / "xgb_model.json")
joblib.dump(encoders, PROCESSED_DIR / "encoders.pkl")
joblib.dump(city_pvout, PROCESSED_DIR / "city_pvout.pkl")
joblib.dump(FEATURE_COLS, PROCESSED_DIR / "feature_cols.pkl")

# Raw holdout with Location and Date intact — agent uses for historical fallback
df_holdout_raw.to_csv(PROCESSED_DIR / "holdout_weather.csv", index=False)

print(
    "Saved: xgb_model.json, encoders.pkl, city_pvout.pkl, feature_cols.pkl, holdout_weather.csv"
)

In [ ]:
# Reload everything to confirm saves are clean
model_loaded = xgb.XGBRegressor()
model_loaded.load_model(PROCESSED_DIR / "xgb_model.json")
encoders_loaded = joblib.load(PROCESSED_DIR / "encoders.pkl")
city_pvout_loaded = joblib.load(PROCESSED_DIR / "city_pvout.pkl")
feature_cols_loaded = joblib.load(PROCESSED_DIR / "feature_cols.pkl")
holdout_loaded = pd.read_csv(
    PROCESSED_DIR / "holdout_weather.csv", parse_dates=["Date"]
)

# Reconstruct X_test and y_test from loaded holdout
holdout_loaded["PVOUT"] = holdout_loaded["Location"].map(city_pvout_loaded)
holdout_loaded = holdout_loaded.dropna(subset=["PVOUT"])

CAT_COLS = ["WindGustDir", "WindDir9am", "WindDir3pm", "RainToday"]
for col in CAT_COLS:
    holdout_loaded[col] = encoders_loaded[col].transform(
        holdout_loaded[col].astype(str)
    )

X_test_reload = holdout_loaded[feature_cols_loaded]
y_test_reload = holdout_loaded["PVOUT"]

y_pred_reload = model_loaded.predict(X_test_reload)
r2_reload = r2_score(y_test_reload, y_pred_reload)

print(f"Model reload R²     : {r2_reload:.4f}")
print(f"Features            : {len(feature_cols_loaded)}")
print(f"Cities in city_pvout: {len(city_pvout_loaded)}")
print(f"Holdout shape       : {holdout_loaded.shape}")
print(
    f"Holdout date range  : {holdout_loaded['Date'].min()} to {holdout_loaded['Date'].max()}"
)